In [1]:
import pandas as pd

train_path = "training_set_VU_DM.csv"
test_path = "test_set_VU_DM.csv"

train_cols = pd.read_csv(train_path, nrows=0).columns.tolist()
test_cols = pd.read_csv(test_path, nrows=0).columns.tolist()

print("Only in train:", sorted(set(train_cols) - set(test_cols)))
print("Only in test:", sorted(set(test_cols) - set(train_cols)))

# 只读建模真正需要的列：test 共有列 + target 所需列
train_usecols = test_cols + ["click_bool", "booking_bool"]

train_df_raw = pd.read_csv(train_path, usecols=train_usecols)
print(train_df_raw.shape)

Only in train: ['booking_bool', 'click_bool', 'gross_bookings_usd', 'position']
Only in test: []
(4958347, 52)


In [2]:
import numpy as np
import pandas as pd

def engineer_lean_features(df, is_train=True, drop_raw_comp=True, copy=False):
    if copy:
        df = df.copy()

    # -------------------------
    # 0. Target for training
    # -------------------------
    if is_train and {"booking_bool", "click_bool"}.issubset(df.columns):
        df["relevance"] = np.select(
            [
                df["booking_bool"] == 1,
                df["click_bool"] == 1
            ],
            [5, 1],
            default=0
        ).astype("int8")

    # -------------------------
    # 1. Competitor summary features
    # -------------------------
    comp_rate_cols = [f"comp{i}_rate" for i in range(1, 9) if f"comp{i}_rate" in df.columns]
    comp_inv_cols = [f"comp{i}_inv" for i in range(1, 9) if f"comp{i}_inv" in df.columns]
    comp_diff_cols = [f"comp{i}_rate_percent_diff" for i in range(1, 9) if f"comp{i}_rate_percent_diff" in df.columns]

    if comp_rate_cols:
        rate = df[comp_rate_cols]

        df["comp_price_advantage_count"] = (rate == 1).sum(axis=1).astype("int8")
        df["comp_price_disadvantage_count"] = (rate == -1).sum(axis=1).astype("int8")
        df["comp_price_equal_count"] = (rate == 0).sum(axis=1).astype("int8")
        df["comp_price_available_count"] = rate.notna().sum(axis=1).astype("int8")
        df["comp_price_missing_count"] = rate.isna().sum(axis=1).astype("int8")

        df["has_competitor_cheaper"] = (df["comp_price_disadvantage_count"] > 0).astype("int8")
        df["has_expedia_cheaper"] = (df["comp_price_advantage_count"] > 0).astype("int8")
        df["has_no_known_cheaper_competitor"] = (
            (df["comp_price_disadvantage_count"] == 0) &
            (df["comp_price_available_count"] > 0)
        ).astype("int8")

    if comp_inv_cols:
        inv = df[comp_inv_cols]

        df["comp_unavailable_count"] = (inv == 1).sum(axis=1).astype("int8")
        df["comp_available_count"] = (inv == 0).sum(axis=1).astype("int8")
        df["comp_inv_missing_count"] = inv.isna().sum(axis=1).astype("int8")
        df["has_comp_unavailable"] = (df["comp_unavailable_count"] > 0).astype("int8")

    # signed price advantage: positive = Expedia cheaper, negative = Expedia more expensive
    signed_cols = []
    for i in range(1, 9):
        rate_col = f"comp{i}_rate"
        diff_col = f"comp{i}_rate_percent_diff"

        if rate_col in df.columns and diff_col in df.columns:
            new_col = f"comp{i}_signed_adv"
            df[new_col] = df[rate_col] * df[diff_col]
            signed_cols.append(new_col)

    if signed_cols:
        signed = df[signed_cols]
        df["comp_signed_adv_mean"] = signed.mean(axis=1).astype("float32")
        df["comp_signed_adv_max"] = signed.max(axis=1).astype("float32")
        df["comp_signed_adv_min"] = signed.min(axis=1).astype("float32")

    if drop_raw_comp:
        raw_comp_cols = []
        for i in range(1, 9):
            raw_comp_cols.extend([
                f"comp{i}_rate",
                f"comp{i}_inv",
                f"comp{i}_rate_percent_diff",
                f"comp{i}_signed_adv"
            ])

        raw_comp_cols = [c for c in raw_comp_cols if c in df.columns]
        df.drop(columns=raw_comp_cols, inplace=True)

    # -------------------------
    # 2. Time features
    # -------------------------
    dt = pd.to_datetime(df["date_time"])
    df["search_month"] = dt.dt.month.astype("int8")
    df["search_day_of_week"] = dt.dt.dayofweek.astype("int8")
    df["search_hour"] = dt.dt.hour.astype("int8")
    df.drop(columns=["date_time"], inplace=True)

    # -------------------------
    # 3. User demand / trip context
    # -------------------------
    df["total_guests"] = (
        df["srch_adults_count"] + df["srch_children_count"]
    ).astype("int16")

    df["is_family"] = (df["srch_children_count"] > 0).astype("int8")

    df["guests_per_room"] = (
        df["total_guests"] / df["srch_room_count"].replace(0, np.nan)
    ).astype("float32")

    df["estimated_total_price"] = (
        df["price_usd"] * df["srch_length_of_stay"] * df["srch_room_count"]
    ).astype("float32")

    df["price_per_person"] = (
        df["estimated_total_price"] / df["total_guests"].replace(0, np.nan)
    ).astype("float32")

    df["same_country_user_hotel"] = (
        df["visitor_location_country_id"] == df["prop_country_id"]
    ).astype("int8")

    # -------------------------
    # 4. Missing indicators
    # -------------------------
    missing_cols = [
        "visitor_hist_starrating",
        "visitor_hist_adr_usd",
        "prop_review_score",
        "prop_location_score2",
        "srch_query_affinity_score",
        "orig_destination_distance"
    ]

    for col in missing_cols:
        if col in df.columns:
            df[f"{col}_missing"] = df[col].isna().astype("int8")

    # -------------------------
    # 5. User-property match features
    # -------------------------
    df["star_signed_diff_from_user_hist"] = (
        df["prop_starrating"] - df["visitor_hist_starrating"]
    ).astype("float32")

    df["star_abs_diff_from_user_hist"] = (
        df["prop_starrating"] - df["visitor_hist_starrating"]
    ).abs().astype("float32")

    df["price_ratio_to_user_hist"] = (
        df["price_usd"] / (df["visitor_hist_adr_usd"] + 1e-6)
    ).astype("float32")

    # -------------------------
    # 6. Historical price features
    # -------------------------
    hist_price = np.where(
        df["prop_log_historical_price"] > 0,
        np.exp(df["prop_log_historical_price"]),
        np.nan
    )

    df["hist_price_missing"] = np.isnan(hist_price).astype("int8")

    df["price_ratio_to_hist"] = (
        df["price_usd"] / (hist_price + 1e-6)
    ).astype("float32")

    # -------------------------
    # 7. Within-search relative features
    # -------------------------
    g = df.groupby("srch_id", sort=False)

    price_median = g["price_usd"].transform("median")
    df["price_ratio_to_srch_median"] = (
        df["price_usd"] / (price_median + 1e-6)
    ).astype("float32")

    df["price_cheap_rank_pct"] = (
        g["price_usd"].rank(pct=True, ascending=False)
    ).astype("float32")

    df["review_high_rank_pct"] = (
        g["prop_review_score"].rank(pct=True, ascending=True)
    ).astype("float32")

    df["star_high_rank_pct"] = (
        g["prop_starrating"].rank(pct=True, ascending=True)
    ).astype("float32")

    df["loc1_high_rank_pct"] = (
        g["prop_location_score1"].rank(pct=True, ascending=True)
    ).astype("float32")

    df["loc2_high_rank_pct"] = (
        g["prop_location_score2"].rank(pct=True, ascending=True)
    ).astype("float32")

    # Neutral fill for rank features when the original variable is missing
    rank_cols = [
        "price_cheap_rank_pct",
        "review_high_rank_pct",
        "star_high_rank_pct",
        "loc1_high_rank_pct",
        "loc2_high_rank_pct"
    ]

    for col in rank_cols:
        df[col] = df[col].fillna(0.5).astype("float32")

    # -------------------------
    # 8. Drop leakage columns
    # -------------------------
    leakage_cols = [
        "position",
        "gross_bookings_usd",
        "click_bool",
        "booking_bool"
    ]

    leakage_cols = [c for c in leakage_cols if c in df.columns]
    df.drop(columns=leakage_cols, inplace=True)

    return df




# #调试（抽sample）
# sample_ids = (
#     train_df_raw["srch_id"]
#     .drop_duplicates()
#     .sample(frac=0.05, random_state=42)
# )

# sample_train = train_df_raw[train_df_raw["srch_id"].isin(sample_ids)].copy()

# processed_sample = engineer_lean_features(
#     sample_train,
#     is_train=True,
#     drop_raw_comp=True,
#     copy=False
# )

# print(processed_sample.shape)
# processed_sample.head()


#完整处理时
train_fe = engineer_lean_features(
    train_df_raw,
    is_train=True,
    drop_raw_comp=True,
    copy=False
)

train_fe.to_parquet("train_features_lean.parquet", index=False)


#读的时候
train_fe = pd.read_parquet("train_features_lean.parquet")